# Лабораторная №1. Работа с данными с помощью Apache Spark

**По методическим указаниям А. А. Друки.** Цель — работа с табличными данными в Spark, Spark SQL и MLlib.

**Структура:** настройка Colab → изучение Spark → полный пример из методички на Fetal Health → дополнение всех пропусков → подбор гиперпараметров → метрики и матрицы ошибок → самостоятельное задание (вариант 4, Smoke Detection).

В Colab последовательно выполните все ячейки. **Обучение моделей не запускалось заранее**: таблицы с результатами будут сформированы после исполнения.

**Датасеты:** [Fetal Health Classification](https://www.kaggle.com/datasets/andrewmvd/fetal-health-classification) и [Smoke Detection](https://www.kaggle.com/datasets/deepcontractor/smoke-detection-dataset). Для каждого предусмотрена загрузка CSV с компьютера; при наличии Kaggle API можно скачать их автоматически.

## 1. Знакомство с Google Colab и Apache Spark

В методичке предлагается открыть Google Colab, создать блокнот и добавлять блоки «+ Код» и «+ Текст». Этот блокнот уже создан и готов к выполнению.

Apache Spark выполняет распределённую обработку данных. **Spark SQL** позволяет анализировать таблицы SQL-запросами, **MLlib** — строить конвейеры машинного обучения. Операции Spark часто *ленивые*: вычисление запускается при вызове действий, например `show()`, `count()` или `collect()`.

### Если возникла ошибка Java heap space

После ошибки JVM может остаться в нерабочем состоянии. Выберите **Среда выполнения → Перезапустить сеанс**, затем выполните ноутбук сверху вниз. Параметр `spark.driver.memory` применяется **только при создании нового SparkContext**, поэтому повторный запуск одной ячейки без перезапуска не поможет. Если Colab не выделяет достаточно RAM, уменьшите значение с `4g` до `2g`, используйте `FULL_GRID=False` и оставьте `parallelism=1`. Для полного перебора параметров может понадобиться среда с увеличенной RAM.

## 2. Установка PySpark и создание SparkSession

Выполняем команды из методички. Для Colab достаточно CPU.

In [ ]:
%pip -q install pyspark

In [ ]:
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import when
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Ограничиваем параллелизм и задаём heap JVM до создания SparkSession.
# После OutOfMemoryError сначала: Среда выполнения → Перезапустить сеанс.
spark = (SparkSession.builder.master("local[2]")
         .appName("Fetal Health Classification")
         .config("spark.driver.memory","4g")
         .config("spark.sql.shuffle.partitions","4")
         .config("spark.default.parallelism","2")
         .getOrCreate())
print("Spark:",spark.version)

## 3. Пример из методички: Fetal Health Classification

Пример предусматривает бинарную классификацию: `fetal_health=1` — норма (0), `2` или `3` — подозрение/патология (1).

### 3.1. Загрузка исходного CSV

Если файл `fetal_health.csv` отсутствует, появится диалог выбора файла. Сначала скачайте CSV с Kaggle. После перезапуска Colab файл может потребоваться загрузить повторно.

In [ ]:
from pathlib import Path
from google.colab import files
if not Path("fetal_health.csv").exists():
    print("Загрузите fetal_health.csv")
    uploaded=files.upload()
    assert "fetal_health.csv" in uploaded, "Нужен файл с точным именем fetal_health.csv"
df=spark.read.csv("fetal_health.csv",header=True,inferSchema=True)
assert "fetal_health" in df.columns, "Не найдена колонка fetal_health"
df.printSchema()
df.show(10,truncate=False)
print("Строк:",df.count())

### 3.2. Первичная обработка и разделение выборок

Воспроизводим преобразование метки из методички и разделение 80/20 с `seed=42`. Удаляем пропуски, если они есть, чтобы VectorAssembler не завершился ошибкой.

In [ ]:
df=df.withColumn("label",when(F.col("fetal_health")==1,F.lit(0.0)).otherwise(F.lit(1.0))).drop("fetal_health")
feature_columns=[c for c in df.columns if c!="label"]
df=df.select(*[F.col(c).cast("double").alias(c) for c in feature_columns],"label").dropna()
# Отсекаем длинную историю преобразований Spark перед CV.
# checkpoint(eager=True) материализует данные и обрезает logical plan.
spark.sparkContext.setCheckpointDir("/tmp/spark_lab1_checkpoints")
df=df.checkpoint(eager=True)
train_data,test_data=df.randomSplit([0.8,0.2],seed=42)
assert train_data.count()>0 and test_data.count()>0
print("Признаков:",len(feature_columns),"обучение:",train_data.count(),"тест:",test_data.count())
df.groupBy("label").count().orderBy("label").show()

### 3.3. Подготовка признаков — VectorAssembler

Все признаки собираются в вектор `features`. Это обязательный вход классификаторов MLlib.

In [ ]:
assembler=VectorAssembler(inputCols=feature_columns,outputCol="features")
assembler.transform(train_data).select("features","label").show(3,truncate=False)

### 3.4. Дописанный фрагмент №1 — создание трёх моделей

В методичке здесь пропуск «Допишите код». Создаём логистическую регрессию, дерево решений и случайный лес.

In [ ]:
lr=LogisticRegression(featuresCol="features",labelCol="label",maxIter=100)
dt=DecisionTreeClassifier(featuresCol="features",labelCol="label",seed=42)
rf=RandomForestClassifier(featuresCol="features",labelCol="label",seed=42)

### 3.5. Сетки гиперпараметров

В методичке для логистической регрессии приведено 36 комбинаций, для дерева 4, для леса 12. При трёхкратной кросс-валидации это **156 обучений**, некоторые очень дорогие (`maxIter=10000`). Поэтому по умолчанию используется ускоренная сетка для проверки работоспособности, а переключатель ниже позволяет включить **точную сетку методички**. Полный вариант может выполняться долго.

In [ ]:
FULL_GRID=False  # True — точные значения из методички, существенно дольше
if FULL_GRID:
    paramGrid_lr=(ParamGridBuilder().addGrid(lr.maxIter,[10,100,1000,10000])
                  .addGrid(lr.regParam,[0.1,0.5,1.0])
                  .addGrid(lr.elasticNetParam,[0.0,0.5,1.0]).build())
    paramGrid_dt=ParamGridBuilder().addGrid(dt.maxDepth,[3,5,9,12]).build()
    paramGrid_rf=(ParamGridBuilder().addGrid(rf.maxDepth,[3,5,9,12])
                  .addGrid(rf.numTrees,[5,11,25]).build())
else:
    paramGrid_lr=(ParamGridBuilder().addGrid(lr.maxIter,[10,100])
                  .addGrid(lr.regParam,[0.1,0.5])
                  .addGrid(lr.elasticNetParam,[0.0,0.5]).build())
    paramGrid_dt=ParamGridBuilder().addGrid(dt.maxDepth,[3,5]).build()
    paramGrid_rf=(ParamGridBuilder().addGrid(rf.maxDepth,[3,5])
                  .addGrid(rf.numTrees,[5,11]).build())
print("Комбинаций:",len(paramGrid_lr),len(paramGrid_dt),len(paramGrid_rf))

### 3.6. Дописанный фрагмент №2 — три CrossValidator

Для каждой модели создаём кросс-валидатор с тремя фолдами. Основная метрика выбора — `f1`.

In [ ]:
evaluator=MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="f1")
cv_lr=CrossValidator(estimator=lr,estimatorParamMaps=paramGrid_lr,evaluator=evaluator,numFolds=3,seed=42,parallelism=1)
cv_dt=CrossValidator(estimator=dt,estimatorParamMaps=paramGrid_dt,evaluator=evaluator,numFolds=3,seed=42,parallelism=1)
cv_rf=CrossValidator(estimator=rf,estimatorParamMaps=paramGrid_rf,evaluator=evaluator,numFolds=3,seed=42,parallelism=1)

### 3.7. Дописанный фрагмент №3 — Pipeline

Сначала собираем вектор признаков, затем запускаем кросс-валидацию.

In [ ]:
pipeline_lr=Pipeline(stages=[assembler,cv_lr])
pipeline_dt=Pipeline(stages=[assembler,cv_dt])
pipeline_rf=Pipeline(stages=[assembler,cv_rf])

### 3.8. Дописанный фрагмент №4 — обучение моделей

При включённой полной сетке обучение может быть долгим. Можно временно оставить `FULL_GRID=False` для проверки.

In [ ]:
# Запускайте только после перезапуска сеанса и выполнения предыдущих ячеек.
# При нехватке RAM отключите FULL_GRID и используйте среду Colab с большим объёмом ОЗУ.
import gc
model_lr=pipeline_lr.fit(train_data)
gc.collect()
model_dt=pipeline_dt.fit(train_data)
gc.collect()
model_rf=pipeline_rf.fit(train_data)
gc.collect()
print("Все три модели обучены")

### 3.9. Дописанные фрагменты №5–7 — оценка и метрики

Создаём функцию, которая вычисляет **Accuracy, weighted Precision, weighted Recall и F1** на тестовой выборке. Отдельно строим матрицу ошибок для каждой модели.

In [ ]:
def evaluate_model(fitted_model,test_df):
    predictions=fitted_model.transform(test_df).cache()
    values=[]
    for metric in ["accuracy","weightedPrecision","weightedRecall","f1"]:
        ev=MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName=metric)
        values.append(ev.evaluate(predictions))
    return predictions,tuple(values)

pred_lr,(accuracy_lr,precision_lr,recall_lr,f1_lr)=evaluate_model(model_lr,test_data)
pred_dt,(accuracy_dt,precision_dt,recall_dt,f1_dt)=evaluate_model(model_dt,test_data)
pred_rf,(accuracy_rf,precision_rf,recall_rf,f1_rf)=evaluate_model(model_rf,test_data)

### 3.10. Дописанный фрагмент №8 — вывод результатов

Сравниваем три модели и визуализируем матрицы ошибок.

In [ ]:
print("Логистическая регрессия - Accuracy = {}, Precision = {}, Recall = {}, F1 = {}".format(accuracy_lr,precision_lr,recall_lr,f1_lr))
print("Дерево решений - Accuracy = {}, Precision = {}, Recall = {}, F1 = {}".format(accuracy_dt,precision_dt,recall_dt,f1_dt))
print("Случайный лес - Accuracy = {}, Precision = {}, Recall = {}, F1 = {}".format(accuracy_rf,precision_rf,recall_rf,f1_rf))
fetal_results=pd.DataFrame([
    {"Модель":"Logistic Regression","Accuracy":accuracy_lr,"Precision":precision_lr,"Recall":recall_lr,"F1":f1_lr},
    {"Модель":"Decision Tree","Accuracy":accuracy_dt,"Precision":precision_dt,"Recall":recall_dt,"F1":f1_dt},
    {"Модель":"Random Forest","Accuracy":accuracy_rf,"Precision":precision_rf,"Recall":recall_rf,"F1":f1_rf}])
display(fetal_results)
fig,axs=plt.subplots(1,3,figsize=(15,4))
for ax,name,pred in zip(axs,["LR","DT","RF"],[pred_lr,pred_dt,pred_rf]):
    pairs=pred.select("label","prediction").toPandas()
    cm=confusion_matrix(pairs.label.astype(int),pairs.prediction.astype(int),labels=[0,1])
    ConfusionMatrixDisplay(cm,display_labels=["Норма","Патология"]).plot(ax=ax,colorbar=False)
    ax.set_title(name)
plt.tight_layout();plt.show()

## 4. Работа с Apache Spark SQL

Цель методички включает SQL-запросы. Создадим временное представление и выполним группировку, агрегирование и сортировку. SQL выполняется внутри Spark.

In [ ]:
df.createOrReplaceTempView("fetal_health_features")
spark.sql("SELECT label, COUNT(*) AS n FROM fetal_health_features GROUP BY label ORDER BY label").show()
feature=feature_columns[0]
spark.sql(f"SELECT label, ROUND(AVG(`{feature}`),3) AS mean_feature FROM fetal_health_features GROUP BY label ORDER BY label").show()
spark.sql(f"SELECT label, `{feature}` FROM fetal_health_features ORDER BY `{feature}` DESC LIMIT 10").show()

## 5. Самостоятельная работа — вариант 4: Smoke Detection

**Задание из методички:** оценить вероятность пожара по показаниям физических датчиков. Целевой столбец — `Fire_Alarm` (0 — нет пожара, 1 — пожар). Повторяем сравнение трёх моделей с подбором гиперпараметров и матрицами ошибок.

Скачайте CSV из [Smoke Detection Dataset](https://www.kaggle.com/datasets/deepcontractor/smoke-detection-dataset) и загрузите его через следующую ячейку.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
csv_files=[n for n in uploaded if n.lower().endswith(".csv")]
assert csv_files,"Нужно загрузить CSV Smoke Detection"
smoke=spark.read.csv(csv_files[0],header=True,inferSchema=True)
# Оригинальный CSV с Kaggle использует пробел: "Fire Alarm".
# Приводим к единому имени, используемому в дальнейших ячейках.
smoke=smoke.withColumnRenamed("Fire Alarm","Fire_Alarm")
assert "Fire_Alarm" in smoke.columns,"Отсутствует Fire_Alarm: выбран неверный CSV"
smoke.printSchema();smoke.show(5)
smoke.groupBy("Fire_Alarm").count().show()

### 5.1. Подготовка и SQL-анализ Smoke Detection

Выбираем числовые признаки, исключаем служебную метку, обрабатываем пропуски и делим данные на обучающую и тестовую части.

In [ ]:
numeric_types={"int","bigint","double","float","smallint"}
# Unnamed: 0 — индекс строки, UTC — время, CNT — счётчик наблюдений.
# Не включаем их в признаки, чтобы не допустить утечки информации.
excluded={"Fire_Alarm","Unnamed: 0","UTC","CNT"}
smoke_features=[c for c,t in smoke.dtypes if t in numeric_types and c not in excluded]
assert smoke_features,"Не найдены числовые признаки"
smoke=smoke.select(*[F.col(c).cast("double").alias(c) for c in smoke_features],F.col("Fire_Alarm").cast("double").alias("label")).dropna(subset=["label"])
smoke.createOrReplaceTempView("smoke_measurements")
spark.sql("SELECT label,COUNT(*) AS n FROM smoke_measurements GROUP BY label ORDER BY label").show()
spark.sql(f"SELECT label,AVG(`{smoke_features[0]}`) AS mean_sensor FROM smoke_measurements GROUP BY label").show()
smoke=smoke.checkpoint(eager=True)
smoke_train,smoke_test=smoke.randomSplit([.8,.2],seed=42)
print("train/test:",smoke_train.count(),smoke_test.count())

### 5.2. Три модели и подбор параметров

Используем тот же подход, что и в учебном примере. Для ускорения обучения `FULL_GRID` сохраняет значение, выбранное выше.

In [ ]:
from pyspark.ml.feature import Imputer
def make_smoke_pipeline(classifier):
    imputed=[c+"_filled" for c in smoke_features]
    return Pipeline(stages=[
        Imputer(inputCols=smoke_features,outputCols=imputed),
        VectorAssembler(inputCols=imputed,outputCol="features"),
        classifier])
smoke_estimators={
    "Logistic Regression":LogisticRegression(maxIter=100),
    "Decision Tree":DecisionTreeClassifier(seed=42),
    "Random Forest":RandomForestClassifier(seed=42,numTrees=11)}
smoke_grids={
    "Logistic Regression":lambda m:(ParamGridBuilder().addGrid(m.maxIter,[10,100,1000,10000] if FULL_GRID else [10,100]).addGrid(m.regParam,[0.1,0.5,1.0] if FULL_GRID else [0.1,0.5]).addGrid(m.elasticNetParam,[0.,.5,1.] if FULL_GRID else [0.,.5]).build()),
    "Decision Tree":lambda m:ParamGridBuilder().addGrid(m.maxDepth,[3,5,9,12] if FULL_GRID else [3,5]).build(),
    "Random Forest":lambda m:(ParamGridBuilder().addGrid(m.maxDepth,[3,5,9,12] if FULL_GRID else [3,5]).addGrid(m.numTrees,[5,11,25] if FULL_GRID else [5,11]).build())}
smoke_predictions={}
smoke_results=[]
for name,est in smoke_estimators.items():
    pipeline=make_smoke_pipeline(est)
    cv=CrossValidator(estimator=pipeline,estimatorParamMaps=smoke_grids[name](est),
                      evaluator=evaluator,numFolds=3,seed=42,parallelism=1)
    fitted=cv.fit(smoke_train)
    pred=fitted.transform(smoke_test).cache()
    smoke_predictions[name]=pred
    metrics={m:MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName=m).evaluate(pred) for m in ["accuracy","weightedPrecision","weightedRecall","f1"]}
    smoke_results.append({"Модель":name,**metrics})
display(pd.DataFrame(smoke_results))

### 5.3. Матрицы ошибок и сравнение

Отдельная матрица ошибок для каждой модели. На несбалансированных данных важно оценивать не только Accuracy, но и Recall для класса «пожар».

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(15,4))
for ax,(name,pred) in zip(axs,smoke_predictions.items()):
    pairs=pred.select("label","prediction").toPandas()
    cm=confusion_matrix(pairs.label.astype(int),pairs.prediction.astype(int),labels=[0,1])
    ConfusionMatrixDisplay(cm,display_labels=["Нет пожара","Пожар"]).plot(ax=ax,colorbar=False)
    ax.set_title(name)
plt.tight_layout();plt.show()
display(pd.DataFrame(smoke_results).sort_values("f1",ascending=False))

## 6. Вывод по лабораторной

После запуска запишите:

1. Какие этапы Spark SQL и MLlib выполнены.
2. Какая модель дала наилучший F1 на Fetal Health и на Smoke Detection.
3. Сколько ложных тревог и пропущенных пожаров видно в матрице ошибок.
4. Как изменились результаты при расширении сетки гиперпараметров.
5. Почему одного Accuracy недостаточно при несбалансированных классах.

**Примечание:** в методичке часть кода дана с пропусками; все обнаруженные пропуски в примере Fetal Health здесь заполнены. Результаты не сфабрикованы и появятся только после запуска.